# 🚀 AI 특강 실습 - 00: 개요 및 환경 설정

---

## 📋 이 노트북에서 볼 것
| 항목 | 내용 |
|------|------|
| **목표** | 강의 전체에서 사용할 환경과 데이터를 준비한다 |
| **예상 실행 시간** | ⏱️ 5~10분 (패키지 설치 포함) |
| **필요한 API 키** | ❌ 불필요 (이 노트북은 API 키 없이 실행 가능) |
| **선택 사항** | OpenAI API 키 있으면 이후 노트북에서 실제 LLM 응답 가능 |

---

## 🗺️ 오늘 강의 노트북 전체 맵

```
00_overview_and_setup        ← 지금 여기 (환경 준비)
01_llm_as_interface          ← LLM을 인터페이스로 보는 관점
02_multimodal_demo           ← 텍스트 너머의 입력 형태
03_embeddings_and_vector_search  ← 왜 검색 계층이 필요한가
04_basic_rag_demo            ← 검색 + LLM 연결 = RAG
05_better_rag_hybrid_and_rerank  ← RAG 품질 개선
06_agentic_search_or_tool_use    ← 모델이 도구를 반복 사용
07_graphrag_concept_demo     ← 관계가 중요할 때의 접근
```

---

## 🎯 오늘 강의의 핵심 메시지

> **"LLM은 단순 챗봇이 아니다. AI 시스템의 공통 인터페이스가 됐다.  
> 이제 중요한 건 모델 이름 암기가 아니라, 문제에 맞는 아키텍처 선택이다."**

각 노트북은 이 메시지의 한 조각을 담고 있습니다.

## 1️⃣ 패키지 설치

강의 전체에서 사용할 패키지를 한 번에 설치합니다.  
**Colab에서는 런타임당 한 번만 실행하면 됩니다.**

In [2]:
# 패키지 설치 (Colab 환경 기준)
# 이미 설치된 경우 빠르게 넘어갑니다
!pip install -q openai sentence-transformers rank-bm25 networkx
print("✅ 패키지 설치 완료")

✅ 패키지 설치 완료


## 2️⃣ 환경 체크

In [4]:
import sys
import os
import platform

print("=" * 55)
print("  🖥️  환경 체크")
print("=" * 55)
print(f"Python: {sys.version.split()[0]}")
print(f"OS: {platform.system()} {platform.release()}")

# GPU 체크
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        print(f"GPU: ✅ {torch.cuda.get_device_name(0)}")
    else:
        print("GPU: ❌ 없음 (CPU 모드로 실행 - 강의에서는 충분합니다)")
except ImportError:
    print("GPU: PyTorch 없음 (CPU 모드)")

# 패키지 체크
packages_to_check = [
    ("openai", "OpenAI SDK"),
    ("sentence_transformers", "로컬 임베딩"),
    ("rank_bm25", "BM25 검색"),
    ("networkx", "그래프 처리"),
    ("pandas", "데이터 처리"),
    ("numpy", "수치 계산"),
    ("sklearn", "ML 유틸"),
]

print("\n패키지 상태:")
for pkg, desc in packages_to_check:
    try:
        __import__(pkg)
        print(f"  ✅ {desc} ({pkg})")
    except ImportError:
        print(f"  ❌ {desc} ({pkg}) - 위 셀을 실행해주세요")

print("=" * 55)

  🖥️  환경 체크
Python: 3.13.5
OS: Darwin 24.3.0
GPU: ❌ 없음 (CPU 모드로 실행 - 강의에서는 충분합니다)

패키지 상태:
  ✅ OpenAI SDK (openai)
  ✅ 로컬 임베딩 (sentence_transformers)
  ✅ BM25 검색 (rank_bm25)
  ✅ 그래프 처리 (networkx)
  ✅ 데이터 처리 (pandas)
  ✅ 수치 계산 (numpy)
  ✅ ML 유틸 (sklearn)


## 3️⃣ API 키 설정 및 모드 선택

### 🔑 API 키가 있는 경우 (권장)
아래 셀에서 API 키를 입력하면 실제 LLM 응답을 볼 수 있습니다.

### 💡 API 키가 없는 경우
**그래도 괜찮습니다!** 로컬 모드로 임베딩/검색은 완전히 동작하고,  
LLM 답변 부분은 사전에 준비된 mock 응답으로 대체됩니다.

In [5]:
import os

# ============================================================
# ✏️ 여기에 API 키를 입력하세요 (없으면 빈 문자열 유지)
# ============================================================
# .env 에서 API 키 로드 (python-dotenv 필요, Colab 에서는 직접 입력 가능)
try:
    from dotenv import load_dotenv
    from pathlib import Path
    for _p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (_p / ".env").exists():
            load_dotenv(_p / ".env"); break
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

# ============================================================
# 모드 자동 감지
# ============================================================
def setup_mode(api_key=""):
    """API 키 유무에 따라 모드를 결정합니다."""
    key = api_key or os.environ.get("OPENAI_API_KEY", "")
    
    if key and key not in ("your-api-key-here", "", "sk-..."):
        os.environ["OPENAI_API_KEY"] = key
        try:
            from openai import OpenAI
            client = OpenAI(api_key=key)
            # 빠른 연결 테스트 (임베딩 1개)
            test = client.embeddings.create(
                input="test", model="text-embedding-3-small"
            )
            print("✅ API 모드: OpenAI 연결 성공!")
            print(f"   사용 모델: gpt-4o-mini / text-embedding-3-small")
            return "api", client
        except Exception as e:
            print(f"⚠️ API 연결 실패: {e}")
            print("🔄 로컬 모드로 전환합니다.")
            return "local", None
    else:
        print("💡 로컬 모드로 실행합니다.")
        print("   임베딩: sentence-transformers (all-MiniLM-L6-v2)")
        print("   LLM: mock 응답 (실제 강의에서는 API 권장)")
        return "local", None

MODE, client = setup_mode(OPENAI_API_KEY)
print(f"\n현재 모드: {'🌐 API' if MODE == 'api' else '💻 LOCAL'}")

✅ API 모드: OpenAI 연결 성공!
   사용 모델: gpt-4o-mini / text-embedding-3-small

현재 모드: 🌐 API


## 4️⃣ 공통 헬퍼 함수 정의

이 함수들은 이후 모든 노트북에서 사용됩니다.  
각 노트북은 독립 실행 가능하지만, 이 코드를 참고용으로 먼저 익혀두세요.

In [6]:
import json
import numpy as np
from typing import List, Dict, Optional

# ──────────────────────────────────────────────
# LLM 호출 헬퍼
# ──────────────────────────────────────────────
def call_llm(prompt, client=None, mode="api",
             model="gpt-4o-mini",
             system="당신은 도움이 되는 AI 어시스턴트입니다.",
             mock_response=None):
    """LLM을 호출합니다. API 실패 시 mock 응답을 반환합니다."""
    if mode == "api" and client is not None:
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7, max_tokens=800
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            print(f"⚠️ LLM 호출 실패: {e}")
    if mock_response:
        return f"[Mock 응답]\n{mock_response}"
    return "[로컬 모드 - API 키를 설정하면 실제 응답을 받을 수 있습니다]"

def call_llm_json(prompt, client=None, mode="api",
                  model="gpt-4o-mini",
                  system="JSON 형식으로만 응답하세요.",
                  mock_response=None):
    """JSON 출력을 요구하는 LLM 호출입니다."""
    if mode == "api" and client is not None:
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.2, max_tokens=1000
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            print(f"⚠️ JSON LLM 호출 실패: {e}")
    if mock_response:
        return mock_response
    return {"error": "로컬 모드"}

# ──────────────────────────────────────────────
# 임베딩 헬퍼
# ──────────────────────────────────────────────
_st_model = None  # 싱글톤 캐싱

def embed_texts(texts, mode="local", client=None,
                api_model="text-embedding-3-small",
                local_model="all-MiniLM-L6-v2"):
    """텍스트 리스트를 numpy 배열로 임베딩합니다."""
    global _st_model
    if mode == "api" and client is not None:
        try:
            vecs = []
            for i in range(0, len(texts), 50):
                batch = texts[i:i+50]
                resp = client.embeddings.create(input=batch, model=api_model)
                vecs.extend([r.embedding for r in resp.data])
            return np.array(vecs, dtype=np.float32)
        except Exception as e:
            print(f"⚠️ API 임베딩 실패 → 로컬 전환: {e}")
    # 로컬 모드
    if _st_model is None:
        print(f"📥 임베딩 모델 로딩 중: {local_model}")
        from sentence_transformers import SentenceTransformer
        _st_model = SentenceTransformer(local_model)
        print("✅ 완료")
    return _st_model.encode(texts, convert_to_numpy=True,
                            show_progress_bar=False).astype(np.float32)

def cosine_sim_batch(query_vec, doc_vecs):
    """쿼리와 문서 배열의 코사인 유사도를 계산합니다."""
    q = np.array(query_vec, dtype=np.float32).flatten()
    D = np.array(doc_vecs, dtype=np.float32)
    q_norm = np.linalg.norm(q)
    if q_norm < 1e-9:
        return np.zeros(len(D))
    d_norms = np.linalg.norm(D, axis=1)
    d_norms = np.where(d_norms < 1e-9, 1e-9, d_norms)
    return (D @ q) / (d_norms * q_norm)

print("✅ 공통 헬퍼 함수 정의 완료")

✅ 공통 헬퍼 함수 정의 완료


## 5️⃣ 샘플 데이터셋 준비

강의 전체에서 사용할 **가상 기업 "테크코어 주식회사"** 의 내부 문서입니다.  

### 데이터 설계 의도
- 📄 **15개 문서**: 보안정책, 릴리즈노트, 회의록, FAQ, 프로세스
- 🔄 **의도적 중복**: 같은 주제의 구버전/신버전 문서 포함 → retrieval 품질 차이 시연
- ⚡ **상충 정보**: 구버전 정책(v2.8)과 신버전(v3.1)이 서로 다른 내용 → 정확한 검색의 중요성 시연

In [7]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# 샘플 문서 데이터 (15개)
# 실제 강의에서는 이 데이터로 모든 노트북 데모를 진행합니다
from helpers.sample_data import SAMPLE_DOCS_OVERVIEW_00 as SAMPLE_DOCS
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

print(f"✅ 샘플 문서 {len(SAMPLE_DOCS)}개 로드 완료")
print("\n문서 목록:")
for doc in SAMPLE_DOCS:
    print(f"  [{doc['doc_id']:25s}] {doc['title']} ({doc['category']})")

✅ 샘플 문서 15개 로드 완료

문서 목록:
  [SEC-POL-003              ] API 키 및 시크릿 관리 정책 v3.1 [현행] (보안정책)
  [SEC-POL-002              ] API 키 관리 정책 v2.8 [구버전 - 비효력] (보안정책)
  [REL-CLOUDSYNC-023        ] CloudSync v2.3 릴리즈 노트 (릴리즈노트)
  [REL-CLOUDSYNC-022        ] CloudSync v2.2 릴리즈 노트 [이전 버전] (릴리즈노트)
  [REL-DATAPULSE-011        ] DataPulse v1.1 릴리즈 노트 (릴리즈노트)
  [WFH-POL-001              ] 원격 근무 정책 v2.0 (HR정책)
  [ONBOARD-ENG-001          ] 개발자 온보딩 체크리스트 (프로세스)
  [MTG-2024Q3-STRATEGY      ] 2024년 3분기 전략 회의록 (회의록)
  [MTG-2024-SEC-REVIEW      ] 보안 정책 개정 검토 회의록 (회의록)
  [FAQ-DATAPULSE-001        ] DataPulse FAQ (FAQ)
  [FAQ-SECURITY-001         ] 보안 정책 FAQ (v3.1 기준) (FAQ)
  [FAQ-HR-001               ] HR 정책 FAQ (FAQ)
  [PROC-DEPLOY-001          ] 운영 배포 체크리스트 v2 (프로세스)
  [PROC-INCIDENT-001        ] 인시던트 대응 절차 v1.3 (프로세스)
  [PROC-CODE-REVIEW-001     ] 코드 리뷰 가이드라인 (프로세스)


## 6️⃣ 데이터 시각화 - 문서셋 한눈에 보기

In [8]:
import pandas as pd
from collections import Counter

# 문서 카테고리 분포
df = pd.DataFrame(SAMPLE_DOCS)[["doc_id", "title", "category", "date"]]
df["내용 길이"] = [len(d["content"]) for d in SAMPLE_DOCS]

print("=== 카테고리별 문서 수 ===")
cat_counts = df["category"].value_counts()
for cat, cnt in cat_counts.items():
    bar = "█" * cnt
    print(f"  {cat:12s}: {bar} ({cnt}개)")

print("\n=== 문서 목록 ===")
display(df[["doc_id", "title", "category", "date", "내용 길이"]])

=== 카테고리별 문서 수 ===
  프로세스        : ████ (4개)
  릴리즈노트       : ███ (3개)
  FAQ         : ███ (3개)
  보안정책        : ██ (2개)
  회의록         : ██ (2개)
  HR정책        : █ (1개)

=== 문서 목록 ===


,doc_id,title,category,date,내용 길이
0,SEC-POL-003,API 키 및 시크릿 관리 정책 v3.1 [현행],보안정책,2024-11-01,447
1,SEC-POL-002,API 키 관리 정책 v2.8 [구버전 - 비효력],보안정책,2024-03-15,218
2,REL-CLOUDSYNC-023,CloudSync v2.3 릴리즈 노트,릴리즈노트,2024-10-20,477
3,REL-CLOUDSYNC-022,CloudSync v2.2 릴리즈 노트 [이전 버전],릴리즈노트,2024-07-15,200
4,REL-DATAPULSE-011,DataPulse v1.1 릴리즈 노트,릴리즈노트,2024-09-05,270
5,WFH-POL-001,원격 근무 정책 v2.0,HR정책,2024-09-01,281
6,ONBOARD-ENG-001,개발자 온보딩 체크리스트,프로세스,2024-08-01,344
7,MTG-2024Q3-STRATEGY,2024년 3분기 전략 회의록,회의록,2024-07-02,396
8,MTG-2024-SEC-REVIEW,보안 정책 개정 검토 회의록,회의록,2024-09-15,335
9,FAQ-DATAPULSE-001,DataPulse FAQ,FAQ,2024-09-10,432


## 7️⃣ 임베딩 모델 사전 로딩 (선택)

로컬 모드에서 임베딩 모델을 미리 다운로드합니다.  
**약 90MB, 최초 1회만 다운로드됩니다.** (이후 캐싱)

In [9]:
# 임베딩 모델 사전 로딩 (로컬 모드 사용 시)
# API 모드를 사용한다면 이 셀을 건너뛰어도 됩니다

if MODE == "local":
    print("📥 로컬 임베딩 모델 로딩 중...")
    print("   모델: all-MiniLM-L6-v2 (약 90MB, 최초 1회 다운로드)")
    try:
        from sentence_transformers import SentenceTransformer
        _st_model = SentenceTransformer("all-MiniLM-L6-v2")
        
        # 테스트 임베딩
        test_vec = _st_model.encode(["테스트 문장"])[0]
        print(f"✅ 로컬 임베딩 모델 준비 완료")
        print(f"   벡터 차원: {len(test_vec)}")
    except Exception as e:
        print(f"❌ 모델 로딩 실패: {e}")
        print("   pip install sentence-transformers 실행 후 재시도")
else:
    print("🌐 API 모드: 로컬 모델 로딩 불필요")
    # API 임베딩 테스트
    try:
        test_resp = client.embeddings.create(
            input="테스트",
            model="text-embedding-3-small"
        )
        print(f"✅ API 임베딩 테스트 완료 (차원: {len(test_resp.data[0].embedding)})")
    except Exception as e:
        print(f"⚠️ API 임베딩 테스트 실패: {e}")

🌐 API 모드: 로컬 모델 로딩 불필요
✅ API 임베딩 테스트 완료 (차원: 1536)


---

## ✅ 준비 완료!

| 체크 | 항목 |
|------|------|
| ✅ | 패키지 설치 |
| ✅ | 환경 체크 |
| ✅ | 모드 설정 (API 또는 Local) |
| ✅ | 공통 헬퍼 함수 정의 |
| ✅ | 샘플 데이터 (15개 문서) |
| ✅ | 임베딩 모델 준비 |

---

## 🎤 강의자 멘트 포인트

> "오늘 사용할 데이터는 가상 기업의 내부 문서입니다.  
> 보안 정책 두 버전(구버전 v2.8, 신버전 v3.1)이 함께 있습니다.  
> 나중에 검색 데모에서 '어떤 버전의 정책이 나오는가'가  
> retrieval quality를 판단하는 좋은 기준이 됩니다."

## 🙋 청중 질문 유도
> - "여러분 회사에도 이런 내부 문서가 있나요? 어떤 형태로 관리되고 있나요?"
> - "AI에게 내부 문서를 '가르치는' 방법이 뭐가 있을까요? (fine-tuning vs RAG?)"

## ➡️ 다음 노트북
**01_llm_as_interface.ipynb** - LLM을 단순 텍스트 생성기가 아닌 인터페이스로 보는 관점